# Pose Control based on Lyapunov Estabilization theory

- Como se faz um projeto não-linear
- A chegada na equação de Lyapunov

## Configuring the environment

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
from coppeliasim_zmqremoteapi_client import RemoteAPIClient
import os

# Parameters of Turtlebot 3
wheel_radius = 0.033
robot_width = 0.287

# Initialize the Remote API
client = RemoteAPIClient()
sim = client.require('sim')

# Open the turtlebot3 scene 
simulation_file = os.getcwd()+'/turtlebot3_pose_estabilization.ttt'
sim.loadScene(simulation_file)

# Use the stepping mode
sim.setStepping(True)   

# Configure the handles
Turtlebot3 = sim.getObjectHandle('/Turtlebot3')
leftMotor = sim.getObjectHandle('/Turtlebot3/left_motor')
rightMotor = sim.getObjectHandle('/Turtlebot3/right_motor')
Goal = sim.getObjectHandle('/ReferenceFrame')

# Error tolerance
min_error = 0.2

# Robot Limits - considering the kinematic model
maxv = 0.26
maxw = 1.82

In [87]:
def draw_robot(x, y, theta, ax, scale=1.0, **kwargs):
    """
    Desenha o contorno do robô no gráfico fornecido.
    
    Parâmetros:
    x, y  : Posição do centro do robô (m)
    theta : Orientação do robô (rad)
    ax    : Objeto Axes do matplotlib onde o robô será desenhado
    scale : Escala do desenho (para ajustar o tamanho visualmente)
    **kwargs: Argumentos opcionais de plotagem (ex: color='b', linestyle='--')
    """
    p = np.zeros((12, 3))
    
    p[:] = [
        [ 1,    1/7,  1/scale],
        [-3/7,  1,    1/scale],
        [-5/7,  6/7,  1/scale],
        [-5/7,  5/7,  1/scale],
        [-3/7,  2/7,  1/scale],
        [-3/7,  0,    1/scale],
        [-3/7, -2/7,  1/scale],
        [-5/7, -5/7,  1/scale],
        [-5/7, -6/7,  1/scale],
        [-3/7, -1,    1/scale],
        [ 1,   -1/7,  1/scale],
        [ 1,    1/7,  1/scale]
    ]
    
    p = scale * p
    
    r = np.array([
        [np.cos(theta),  np.sin(theta)],
        [-np.sin(theta), np.cos(theta)],
        [x,              y]
    ])
    
    p_transf = np.dot(p, r)
    
    X_plot = p_transf[:, 0]
    Y_plot = p_transf[:, 1]
    
    if 'color' not in kwargs and 'c' not in kwargs:
        kwargs['color'] = 'blue'
        
    ax.plot(X_plot, Y_plot, **kwargs)

# Normalize angle to the range [-pi,pi)
def normalize_angle(angle):
    return np.mod(angle+np.pi, 2*np.pi) - np.pi

## Simulation Loop - Aicardi

In [ ]:
# Initialize the simulation
sim.startSimulation()

# Control gains
gamma = 1*2
h = 1/1.5
k = 1*2

while True:

    # Capture the robot pose from simulation
    TBPos=sim.getObjectPosition(Turtlebot3,-1)
    TBXOri=sim.getObjectOrientation(Turtlebot3,-1)
    qTurtlebot = np.array([TBPos[0], TBPos[1], TBXOri[2]])

    # Capture the goal pose
    Goal_Pos = sim.getObjectPosition(Goal,-1)
    Goal_Ori = sim.getObjectOrientation(Goal,-1)
    qGoal = np.array([Goal_Pos[0], Goal_Pos[1],Goal_Ori[2]])

    # Global States
    dx, dy, dth = qGoal - qTurtlebot

    # Transform to Aicardi states
    e = math.sqrt(dx**2 + dy**2)
    alpha = normalize_angle(np.arctan2(dy,dx) - qTurtlebot[2])
    theta_aicardi = normalize_angle(qGoal[2] - np.arctan2(dy,dx))

    # Stopping condition
    if e < min_error:
        # Goal achieved
        print("Alvo alcançado!")
        sim.stopSimulation()
        break

    # Calculate linear and angular velocities from (6) and (9)
    # u = gamma * cos(alpha) * e
    v = gamma * math.cos(alpha) * e
    
    # omega = k*alpha + gamma * (cos(alpha)*sin(alpha)/alpha) * (alpha + h*theta)
    omega = k * alpha + gamma * ((math.cos(alpha) * math.sin(alpha))/alpha) * (alpha + h * theta_aicardi)

    # Saturation Limits
    v = max(min(v, maxv), -maxv)
    omega = max(min(omega, maxw), -maxw) 

    #  DDMR Inverse Kinematics
    w_right = (v + (omega * robot_width / 2)) / wheel_radius
    w_left  = (v - (omega * robot_width / 2)) / wheel_radius

    # Send commands
    sim.setJointTargetVelocity(leftMotor, w_left)
    sim.setJointTargetVelocity(rightMotor, w_right)

    print(f"Erro: {e} | Alpha: {alpha} | Theta: {theta_aicardi}")
    print(f"v: {v}, omega: {omega}\n")
    # Step the simulation
    sim.step()

## Simulation Loop - Benbouabdallah

In [ ]:
# Initialize the Remote API
client = RemoteAPIClient()
sim = client.require('sim')

# Use the stepping mode
sim.setStepping(True)   

# Configure the handles
Turtlebot3 = sim.getObjectHandle('/Turtlebot3')
leftMotor = sim.getObjectHandle('/Turtlebot3/left_motor')
rightMotor = sim.getObjectHandle('/Turtlebot3/right_motor')
Goal = sim.getObjectHandle('/ReferenceFrame')

# Initialize the simulation
sim.startSimulation()

# Control gains
Kv = 2.07
Kw = 1.49

Dd = 0.2

while True:

    # Capture the robot pose from simulation
    TBPos=sim.getObjectPosition(Turtlebot3,-1)
    TBXOri=sim.getObjectOrientation(Turtlebot3,-1)
    qTurtlebot = np.array([TBPos[0], TBPos[1], TBXOri[2]])

    # Capture the goal pose
    Goal_Pos = sim.getObjectPosition(Goal,-1)
    Goal_Ori = sim.getObjectOrientation(Goal,-1)
    qGoal = np.array([Goal_Pos[0], Goal_Pos[1],Goal_Ori[2]])

    # Global States
    dx, dy, dth = qGoal - qTurtlebot
    # Transform to Benbouabdallah states
    e = math.sqrt(dx**2 + dy**2)
    alpha = normalize_angle(np.arctan2(dy,dx) - qTurtlebot[2])
    theta_aicardi = normalize_angle(qGoal[2] - np.arctan2(dy,dx))
    ed = Dd - e

    # Stopping condition
    if e < Dd:
        # Goal achieved
        print("Alvo alcançado!")
        sim.stopSimulation()
        break

    # Calculate linear and angular velocities from (14)
    # v_tt = -Kv * ed * math.cos(alpha)
    v_tt = - Kv * ed * math.cos(alpha)
    
    # omega = k*alpha + gamma * (cos(alpha)*sin(alpha)/alpha) * (alpha + h*theta)
    omega_tt = -Kw * alpha - (v_tt/e) * math.sin(alpha)

    # Saturation Limits
    v_tt = max(min(v_tt, maxv), -maxv)
    omega_tt = max(min(omega_tt, maxw), -maxw) 


    #  DDMR Inverse Kinematics
    w_right = (v_tt + (omega_tt * robot_width / 2)) / wheel_radius
    w_left  = (v_tt - (omega_tt * robot_width / 2)) / wheel_radius

    # Send commands
    sim.setJointTargetVelocity(leftMotor, w_left)
    sim.setJointTargetVelocity(rightMotor, w_right)

    print(f"Erro: {e} | Alpha: {alpha} | Theta: {theta_aicardi}")
    print(f"v: {v}, omega: {omega}\n")
    # Step the simulation
    sim.step()